# GAC Motors Import Analysis to UAE (Last 5 Years)
## Forecasting with FBProphet and XGBoost

This notebook extracts GAC Motors import units to UAE from CAAM and Jato Dynamics data, visualizes historical trends, and predicts future imports using FBProphet and XGBoost models, comparing RMSE to select the best model.


## Section 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings
from datetime import datetime, timedelta
import os

warnings.filterwarnings('ignore')

# Set styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("All required libraries imported successfully!")

## Section 2: Load CAAM and Jato Dynamics Data
**Note:** This notebook creates realistic synthetic data. In production, replace this with actual CAAM and Jato Dynamics CSV/Excel files.

In [ ]:
# Create synthetic CAAM data (Central Agency for Standardized Measurements)
# Simulating last 5 years of GAC Motors import units to UAE
np.random.seed(42)

# Generate date range for last 5 years (monthly)
end_date = datetime(2026, 5, 31)
start_date = end_date - timedelta(days=365*5)
date_range = pd.date_range(start=start_date, end=end_date, freq='MS')

# Base trend with seasonality
base_units = 500
trend = np.linspace(0, 300, len(date_range))
seasonality = 150 * np.sin(np.linspace(0, 10*np.pi, len(date_range)))
noise = np.random.normal(0, 50, len(date_range))
gac_units_caam = np.maximum(base_units + trend + seasonality + noise, 100).astype(int)

caam_data = pd.DataFrame({
    'Date': date_range,
    'Company': 'GAC Motors',
    'Country': 'UAE',
    'Import_Units': gac_units_caam,
    'Source': 'CAAM'
})

# Create Jato Dynamics data (similar pattern with slight variations)
jato_units = np.maximum(base_units + trend*0.9 + seasonality*1.1 + np.random.normal(0, 60, len(date_range)), 80).astype(int)

jato_data = pd.DataFrame({
    'Date': date_range,
    'Company': 'GAC Motors',
    'Country': 'UAE',
    'Import_Units': jato_units,
    'Source': 'Jato Dynamics'
})

print("CAAM Data Sample (First 10 rows):")
print(caam_data.head(10))
print(f"\nCAAM Data Shape: {caam_data.shape}")
print("\n" + "="*60)
print("\nJato Dynamics Data Sample (First 10 rows):")
print(jato_data.head(10))
print(f"\nJato Data Shape: {jato_data.shape}")

## Section 3: Clean and Merge Import Data

In [ ]:
# Combine both data sources
combined_data = pd.concat([caam_data, jato_data], ignore_index=True)

# Data cleaning
combined_data['Date'] = pd.to_datetime(combined_data['Date'])

# Check for missing values
print("Missing values in combined data:")
print(combined_data.isnull().sum())
print("\n" + "="*60)

# Average the units from both sources for the same month
merged_data = combined_data.groupby('Date').agg({
    'Import_Units': 'mean',
    'Company': 'first',
    'Country': 'first'
}).reset_index()

merged_data['Import_Units'] = merged_data['Import_Units'].astype(int)
merged_data = merged_data.sort_values('Date').reset_index(drop=True)

print("\nMerged and Cleaned Data:")
print(merged_data.head(10))
print(f"\nTotal records: {len(merged_data)}")
print(f"Date range: {merged_data['Date'].min()} to {merged_data['Date'].max()}")
print(f"\nBasic Statistics:")
print(merged_data['Import_Units'].describe())

## Section 4: Visualize Historical GAC Motors Imports

In [ ]:
# Matplotlib visualization
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Line chart
axes[0].plot(merged_data['Date'], merged_data['Import_Units'], linewidth=2, color='#2E86AB', marker='o', markersize=4)
axes[0].fill_between(merged_data['Date'], merged_data['Import_Units'], alpha=0.3, color='#2E86AB')
axes[0].set_title('GAC Motors Import Units to UAE - Last 5 Years (Trend)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Import Units', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Monthly boxplot to show seasonality
merged_data['Month'] = merged_data['Date'].dt.month
merged_data['Month_Name'] = merged_data['Date'].dt.strftime('%b')
monthly_data = merged_data.sort_values('Month')
axes[1].bar(monthly_data['Month_Name'], monthly_data['Import_Units'], color='#A23B72', alpha=0.7)
axes[1].set_title('GAC Motors Import Units by Month - Seasonality Analysis', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month', fontsize=12)
axes[1].set_ylabel('Average Import Units', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nHistorical Data Visualization Summary:")
print(f"Min Import Units: {merged_data['Import_Units'].min()}")
print(f"Max Import Units: {merged_data['Import_Units'].max()}")
print(f"Average Import Units: {merged_data['Import_Units'].mean():.0f}")

## Section 5: Prepare Time Series for Prophet

In [ ]:
# Prepare data for Prophet (requires 'ds' and 'y' columns)
prophet_df = merged_data[['Date', 'Import_Units']].copy()
prophet_df.columns = ['ds', 'y']

# Split into training and test sets (80-20 split)
split_point = int(len(prophet_df) * 0.8)
train_df = prophet_df[:split_point].copy()
test_df = prophet_df[split_point:].copy()

print(f"Total records: {len(prophet_df)}")
print(f"Training set: {len(train_df)} records ({train_df['ds'].min().date()} to {train_df['ds'].max().date()})")
print(f"Test set: {len(test_df)} records ({test_df['ds'].min().date()} to {test_df['ds'].max().date()})")
print("\n" + "="*60)
print("\nTrain-Test Split Preview:")
print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")

## Section 6: Train and Evaluate Prophet Forecast

In [ ]:
try:
    from prophet import Prophet
    prophet_available = True
except ImportError:
    print("⚠️ Prophet not available. Installing...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'pystan==2.14.10', '-q'])
    subprocess.check_call(['pip', 'install', 'prophet', '-q'])
    from prophet import Prophet
    prophet_available = True

# Train Prophet model
print("Training Prophet model...")
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95,
    growth='linear'
)

prophet_model.fit(train_df)

# Generate forecast for test period
future_periods = len(test_df)
future = prophet_model.make_future_dataframe(periods=future_periods)
forecast = prophet_model.predict(future)

# Extract predictions for test period
forecast_test = forecast[forecast['ds'].isin(test_df['ds'])][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)

# Calculate RMSE for Prophet
y_true = test_df['y'].values
y_pred = forecast_test['yhat'].values
prophet_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
prophet_mae = mean_absolute_error(y_true, y_pred)
prophet_r2 = r2_score(y_true, y_pred)

print(f"\n{'='*60}")
print("PROPHET MODEL PERFORMANCE")
print(f"{'='*60}")
print(f"RMSE: {prophet_rmse:.2f} units")
print(f"MAE: {prophet_mae:.2f} units")
print(f"R² Score: {prophet_r2:.4f}")

# Visualization - Prophet forecast
fig, ax = plt.subplots(figsize=(15, 7))

# Plot training data
ax.plot(train_df['ds'], train_df['y'], 'o-', label='Training Data', color='#2E86AB', markersize=4)

# Plot test data
ax.plot(test_df['ds'], test_df['y'], 'o-', label='Actual Test Data', color='#06A77D', markersize=4, linewidth=2)

# Plot Prophet forecast
ax.plot(forecast_test['ds'], forecast_test['yhat'], 's--', label='Prophet Forecast', color='#D62828', markersize=5, linewidth=2)

# Add confidence interval
ax.fill_between(forecast_test['ds'], forecast_test['yhat_lower'], forecast_test['yhat_upper'], 
                alpha=0.2, color='#D62828', label='95% Confidence Interval')

ax.set_title('GAC Motors Imports - Prophet Forecast vs Actual', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Import Units', fontsize=12)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nProphet forecast visualization complete!")

## Section 7: Prepare Features for XGBoost

In [ ]:
# Create features for XGBoost
xgb_df = merged_data[['Date', 'Import_Units']].copy()
xgb_df['Year'] = xgb_df['Date'].dt.year
xgb_df['Month'] = xgb_df['Date'].dt.month
xgb_df['Quarter'] = xgb_df['Date'].dt.quarter
xgb_df['DayOfYear'] = xgb_df['Date'].dt.dayofyear
xgb_df['WeekOfYear'] = xgb_df['Date'].dt.isocalendar().week

# Create lag features (previous 1, 3, 6, 12 months)
xgb_df['Lag_1'] = xgb_df['Import_Units'].shift(1)
xgb_df['Lag_3'] = xgb_df['Import_Units'].shift(3)
xgb_df['Lag_6'] = xgb_df['Import_Units'].shift(6)
xgb_df['Lag_12'] = xgb_df['Import_Units'].shift(12)

# Create rolling statistics
xgb_df['Rolling_Mean_3'] = xgb_df['Import_Units'].rolling(window=3).mean()
xgb_df['Rolling_Mean_6'] = xgb_df['Import_Units'].rolling(window=6).mean()
xgb_df['Rolling_Std_3'] = xgb_df['Import_Units'].rolling(window=3).std()

# Drop NaN values resulting from lag and rolling features
xgb_df = xgb_df.dropna().reset_index(drop=True)

# Prepare features and target
X = xgb_df[['Year', 'Month', 'Quarter', 'DayOfYear', 'WeekOfYear', 
             'Lag_1', 'Lag_3', 'Lag_6', 'Lag_12', 'Rolling_Mean_3', 'Rolling_Mean_6', 'Rolling_Std_3']]
y = xgb_df['Import_Units']

# Split into train and test
split_xgb = int(len(X) * 0.8)
X_train, X_test = X[:split_xgb], X[split_xgb:]
y_train, y_test = y[:split_xgb], y[split_xgb:]

print("XGBoost Feature Engineering Complete")
print(f"Total samples: {len(X)}")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print("\nFeatures created:")
print(X.columns.tolist())
print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Section 8: Train and Evaluate XGBoost Regression

In [ ]:
# Train XGBoost model
print("Training XGBoost model...")
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=0
)

xgb_model.fit(X_train, y_train, verbose=False)

# Make predictions
y_pred_xgb = xgb_model.predict(X_test)

# Calculate metrics
xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_r2 = r2_score(y_test, y_pred_xgb)

print(f"\n{'='*60}")
print("XGBOOST MODEL PERFORMANCE")
print(f"{'='*60}")
print(f"RMSE: {xgb_rmse:.2f} units")
print(f"MAE: {xgb_mae:.2f} units")
print(f"R² Score: {xgb_r2:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*60)
print("TOP 10 IMPORTANT FEATURES:")
print("="*60)
print(feature_importance.head(10).to_string(index=False))

# Visualization - XGBoost predictions vs actual
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Actual vs Predicted
axes[0].plot(range(len(y_test)), y_test.values, 'o-', label='Actual', color='#06A77D', markersize=5, linewidth=2)
axes[0].plot(range(len(y_pred_xgb)), y_pred_xgb, 's--', label='XGBoost Prediction', color='#D62828', markersize=5, linewidth=2)
axes[0].set_title('XGBoost: Actual vs Predicted Import Units', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Test Sample Index', fontsize=12)
axes[0].set_ylabel('Import Units', fontsize=12)
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Feature Importance
top_features = feature_importance.head(10)
axes[1].barh(top_features['Feature'], top_features['Importance'], color='#2E86AB', alpha=0.7)
axes[1].set_title('Top 10 Feature Importance in XGBoost Model', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importance Score', fontsize=12)
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Section 9: Compare RMSE and Select Best Model

In [ ]:
# Model Comparison
comparison_df = pd.DataFrame({
    'Model': ['FBProphet', 'XGBoost'],
    'RMSE': [prophet_rmse, xgb_rmse],
    'MAE': [prophet_mae, xgb_mae],
    'R² Score': [prophet_r2, xgb_r2]
})

print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Determine best model
best_model = comparison_df.loc[comparison_df['RMSE'].idxmin(), 'Model']
min_rmse = comparison_df['RMSE'].min()

print(f"\n✅ BEST MODEL: {best_model}")
print(f"✅ MINIMUM RMSE: {min_rmse:.2f} units")
print(f"\nDifference in RMSE: {abs(prophet_rmse - xgb_rmse):.2f} units")
print(f"Improvement: {(abs(prophet_rmse - xgb_rmse) / max(prophet_rmse, xgb_rmse)) * 100:.2f}%")

# Visualization - Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# RMSE Comparison
models = comparison_df['Model'].tolist()
rmse_values = comparison_df['RMSE'].tolist()
colors = ['#D62828' if m == best_model else '#2E86AB' for m in models]
axes[0].bar(models, rmse_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[0].set_title('Model Comparison: RMSE (Lower is Better)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('RMSE (units)', fontsize=12)
for i, (model, rmse) in enumerate(zip(models, rmse_values)):
    axes[0].text(i, rmse + 5, f'{rmse:.2f}', ha='center', fontweight='bold', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# R² Score Comparison
r2_values = comparison_df['R² Score'].tolist()
axes[1].bar(models, r2_values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[1].set_title('Model Comparison: R² Score (Higher is Better)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_ylim([0, 1])
for i, (model, r2) in enumerate(zip(models, r2_values)):
    axes[1].text(i, r2 + 0.02, f'{r2:.4f}', ha='center', fontweight='bold', fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section 10: Forecast Future Import Units

In [ ]:
# Use the best model for future forecasting
future_periods = 12  # Forecast next 12 months

if best_model == 'FBProphet':
    print(f"\n{'='*60}")
    print(f"FUTURE FORECAST USING {best_model.upper()}")
    print(f"{'='*60}")
    
    # Prophet forecast for next 12 months
    future = prophet_model.make_future_dataframe(periods=future_periods)
    future_forecast = prophet_model.predict(future)
    future_forecast = future_forecast[future_forecast['ds'] > prophet_df['ds'].max()][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].reset_index(drop=True)
    
    # Create future dates
    last_date = merged_data['Date'].max()
    future_dates = pd.date_range(start=last_date + timedelta(days=31), periods=future_periods, freq='MS')
    
    forecast_results = pd.DataFrame({
        'Date': future_dates,
        'Forecasted_Units': future_forecast['yhat'].values.astype(int),
        'Lower_Bound': future_forecast['yhat_lower'].values.astype(int),
        'Upper_Bound': future_forecast['yhat_upper'].values.astype(int)
    })
    
else:  # XGBoost
    print(f"\n{'='*60}")
    print(f"FUTURE FORECAST USING {best_model.upper()}")
    print(f"{'='*60}")
    
    # Create future features for XGBoost
    last_date = merged_data['Date'].max()
    future_dates = pd.date_range(start=last_date + timedelta(days=31), periods=future_periods, freq='MS')
    
    future_features = []
    for date in future_dates:
        future_features.append({
            'Year': date.year,
            'Month': date.month,
            'Quarter': date.quarter,
            'DayOfYear': date.dayofyear,
            'WeekOfYear': date.isocalendar()[1],
            'Lag_1': y_test.iloc[-1] if len(y_test) > 0 else y_train.iloc[-1],
            'Lag_3': y_test.iloc[-3] if len(y_test) >= 3 else y_train.iloc[-3],
            'Lag_6': y_test.iloc[-6] if len(y_test) >= 6 else y_train.iloc[-6],
            'Lag_12': y_train.iloc[-12] if len(y_train) >= 12 else y_train.iloc[0],
            'Rolling_Mean_3': y.iloc[-3:].mean(),
            'Rolling_Mean_6': y.iloc[-6:].mean(),
            'Rolling_Std_3': y.iloc[-3:].std()
        })
    
    X_future = pd.DataFrame(future_features)
    future_predictions = xgb_model.predict(X_future).astype(int)
    
    # Estimate confidence intervals (using std of residuals)
    residuals = y_test.values - y_pred_xgb
    std_residual = np.std(residuals)
    
    forecast_results = pd.DataFrame({
        'Date': future_dates,
        'Forecasted_Units': future_predictions,
        'Lower_Bound': (future_predictions - 1.96 * std_residual).astype(int),
        'Upper_Bound': (future_predictions + 1.96 * std_residual).astype(int)
    })

print("\nFORECASTED GAC MOTORS IMPORT UNITS (Next 12 Months):")
print(forecast_results.to_string(index=False))

# Visualization - Future Forecast
fig, ax = plt.subplots(figsize=(15, 7))

# Historical data
ax.plot(merged_data['Date'], merged_data['Import_Units'], 'o-', label='Historical Data', 
        color='#2E86AB', markersize=5, linewidth=2)

# Forecast
ax.plot(forecast_results['Date'], forecast_results['Forecasted_Units'], 's--', label=f'{best_model} Forecast',
        color='#D62828', markersize=6, linewidth=2.5)

# Confidence interval
ax.fill_between(forecast_results['Date'], forecast_results['Lower_Bound'], forecast_results['Upper_Bound'],
               alpha=0.2, color='#D62828', label='95% Confidence Interval')

# Formatting
ax.axvline(x=merged_data['Date'].max(), color='gray', linestyle=':', linewidth=2, alpha=0.7, label='Forecast Start')
ax.set_title(f'GAC Motors Import Units - {best_model} Forecast (Next 12 Months)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Import Units', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Analysis Complete!")